# Important ANN architectures
* after the basic multi-layer perceptron (MLP), several specialized architectures were developed
* each one builds different assumptions about the *structure* of the data into the network itself


## Recap: MLP
* the plain MLP treats its input as an unordered vector of numbers (*features*)
* every input is connected to every neuron of the next layer (*fully connected*)
* no built-in notion of order, locality or context


* works well for fixed-size feature vectors, e.g. [functionals](http://blog.syntheticspeech.de/2022/02/16/kinds-of-machine-learning/) computed over a whole utterance
* less suited for raw sequences (audio, text) or images, where the *position* of a pattern matters


### MLP: parameter explosion
* every input unit connects to every hidden unit &rarr; the number of weights grows with input size &times; hidden size
* e.g. a modest 224x224 RGB image has ~150,000 input values; connecting it to just 1,000 hidden neurons already needs ~150 million weights in that *one* layer
* impractical to train, easy to overfit, and it ignores that nearby pixels (or spectrogram bins) are related


## Why not just use MLPs for everything?
Speech (like most real-world signals) has structure:
* local patterns in the spectrogram (e.g. formant transitions)
* a temporal order: what happens depends on what happened before
* long-range dependencies, e.g. context from the beginning of a sentence

This motivates architectures that build these assumptions into the network.


## CNN (Convolutional Neural Networks)
* instead of connecting every input to every neuron, a small *kernel* (filter) slides over the input
* the same weights are reused at every position (*weight sharing*)
    * far fewer parameters than a fully connected layer
    * *translation invariance*: a pattern is detected wherever it occurs
* stacking convolution + *pooling* (downsampling) layers builds up increasingly abstract features


<img src=images/cnn.png width=90%>

In speech, CNNs are typically applied to
* spectrograms, treated like images (time x frequency), or
* directly to the raw waveform, e.g. [SincNet](https://arxiv.org/abs/1808.00158)

They are well suited to detect local time-frequency patterns, e.g. formant transitions, no matter *where* in the utterance they occur.


## RNN (Recurrent Neural Networks)
* process a sequence step by step, keeping a *hidden state* that is passed on from one step to the next
* the same weights are reused at every time step (weight sharing, but now over *time* instead of space)
* can in principle handle sequences of arbitrary length, unlike an MLP or CNN with a fixed-size input


<img src=images/onetomany.png width=95%>

Different topologies of the same idea:
* one-to-many, e.g. captioning
* many-to-one, e.g. classifying a whole utterance
* many-to-many, e.g. speech recognition, translation


### Vanishing gradients &rarr; LSTM / GRU
* plain RNNs struggle with long-range dependencies: gradients shrink (or explode) when backpropagated through many time steps
* LSTM (*Long Short-Term Memory*) and GRU (*Gated Recurrent Unit*) add learned *gates* that decide what to remember, forget and output
* this made RNNs practical for longer sequences, and they were the backbone of speech recognition/synthesis for many years, before Transformers took over


In speech, RNNs/LSTMs
* model an utterance frame by frame, e.g. a sequence of MFCCs
* were used in earlier end-to-end ASR/TTS systems
* are often combined with CNNs: extract local features first (CNN), then model them sequentially (RNN)


## Attention
* problem with RNNs: the whole sequence has to be squeezed through a single hidden state
    * information about early inputs can get diluted by the time you reach the end
* attention lets the model directly look back at *all* previous states, and learns how much *weight* to give each of them for the current prediction
* originally added on top of RNNs (e.g. for translation), later became the core building block of its own architecture


<img src=images/attention.png width=65%>

## Transformer
* built entirely out of (self-)attention, without any recurrence
* no step-by-step recurrence &rarr; all positions can be processed *in parallel*
    * much faster to train on large amounts of data than RNNs
* attention itself has no notion of order, so a *positional encoding* is added to the input to tell the model the order of positions


<img src=images/transformer.png width=35%>

### Does "no recurrence" mean fixed-length input?
* no &mdash; self-attention works over however many positions you feed in, just like an RNN
* what is removed is the *recurrence* (the step-by-step hidden state), not the ability to handle variable length
* the real practical limit is compute: attention costs O(n&sup2;) in sequence length, so very long sequences get expensive fast
* that is why models use a fixed *context window* in practice (e.g. Whisper: 30s chunks) &mdash; an engineering choice, not an architectural requirement


In speech, Transformers
* are the basis of most current state-of-the-art speech models, e.g. wav2vec 2.0, HuBERT, Whisper
* are often pre-trained *self-supervised* on huge amounts of unlabelled audio, then fine-tuned on a small labelled dataset for the actual task (emotion, ASR, ...)
* the *Conformer* architecture combines convolution (local patterns) with self-attention (long-range context), popular in ASR


## Overview
* **MLP**: fixed-size feature vector, no structure assumed
* **CNN**: local, translation-invariant patterns (in space or time)
* **RNN/LSTM**: sequential, step-by-step processing, arbitrary length
* **Transformer**: all positions at once via self-attention, best at long-range context, but needs a lot of data
